In [ ]:
from transformers import AutoModelForCausalLM , AutoTokenizer , TrainingArguments,Trainer
from peft import LoraConfig,get_peft_model,TaskType
from datasets import load_dataset

In [ ]:
model_name = "Qwen/Qwen2.5-3B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokernizer.pad_token is None:
    tokernizer.pad_token = tokernizer.eos_token

In [ ]:
import zipfile

with zipfile.ZipFile('/content/non_instruct_cardio.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path="/content/content/tinyllama-lora/checkpoint-300"

In [ ]:
pip install --upgrade torchao>=0.16.0

In [ ]:
non_instruct_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", load_in_8bit=True)

In [ ]:
prompt = "Which profile has congestion and impaired tissue perfusion?"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = non_instruct_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

In [ ]:
dataset

In [ ]:
def format_row(example):
    question = example["Context"]
    answer = example["Response"]
    example["Text"] = f"[Context] {question} [/Response] {answer}"
    return example

In [ ]:
formatted_dataset = dataset.map(format_row)

In [ ]:
formatted_dataset

In [ ]:
print(formatted_dataset[0]["Text"])

In [ ]:
import pandas as pd
df = pd.DataFrame(dataset)

In [ ]:
df

In [ ]:
df.to_csv("mental_health_counseling_conversations.csv", index=False)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv",split="train")
dataset

In [ ]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [ ]:
dataset = dataset.map(format_example)

In [ ]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
instruction_model = get_peft_model(non_instruct_model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./cardio-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=instruction_model,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
# trainer.train()